# 01 - Anatomy of a Parquet File

## Question

How is a Parquet file physically organized?

Can we inspect a Parquet file and validate the concepts discussed in `docs/lakehouse/03-anatomy-of-a-parquet-file.md`?

---

## Hypothesis

A Parquet file should contain:

- File-level metadata (Footer)
- Schema
- Row Groups
- Compression information
- Encodings
- Statistics

Instead of relying on documentation, we will inspect a real Parquet file and verify these concepts ourselves.

In [25]:
import pyarrow.parquet as pq
from pathlib import Path

## Locate a Parquet File

Copy the users dataset under `datasets/processed`. We'll inspect the user dimension dataset that we generated earlier.

In [26]:
parquet_dir = Path("../../../datasets/processed/")
print(parquet_dir.resolve())

/Users/swapnilsankla/study/data-platform-lab/datasets/processed


In [27]:
parquet_files = list(parquet_dir.glob("*.parquet"))

print(f"Found {len(parquet_files)} parquet file(s).")

for file in parquet_files:
    print(file.name)

Found 1 parquet file(s).
users.snappy.parquet


Spark writes datasets as directories.

Each directory may contain one or more Parquet files.

We'll inspect the first file.

In [28]:
pf = pq.ParquetFile(parquet_files[0])

## File Metadata

Let's inspect the high-level metadata.

In [ ]:
metadata = pf.metadata

print("Rows:", metadata.num_rows)
print("Columns:", metadata.num_columns)
print("Row Groups:", metadata.num_row_groups)
print("Format Version:", metadata.format_version)
print("Created By:", metadata.created_by)


Rows: 1848737
Columns: 2
Row Groups: 1
Format Version: 1.0
Created By: parquet-mr version 1.13.1 (build db4183109d5b734ec5930d870cdae161e408ddba)
Serialised size: 603


In [30]:
print(pf.schema)

required group field_id=-1 spark_schema {
  optional binary field_id=-1 user_id (String);
  required binary field_id=-1 country (String);
}



In [31]:
for i in range(metadata.num_row_groups):
    rg = metadata.row_group(i)

    print(f"Row Group {i}")
    print("----------------------")
    print("Rows:", rg.num_rows)
    print("Size:", rg.total_byte_size)
    print()

Row Group 0
----------------------
Rows: 1848737
Size: 24736623



In [32]:
rg = metadata.row_group(0)

for i in range(rg.num_columns):
    column = rg.column(i)

    print("=" * 60)
    print(column.path_in_schema)
    print("=" * 60)

    print(column)
    print()

user_id
  file_offset: 4
  file_path: 
  physical_type: BYTE_ARRAY
  num_values: 1848737
  path_in_schema: user_id
  is_stats_set: True
  statistics:
      has_min_max: True
      min: 100963605
      max: 95388363
      null_count: 0
      distinct_count: None
      num_values: 1848737
      physical_type: BYTE_ARRAY
      logical_type: String
      converted_type (legacy): UTF8
  geo_statistics:
    None
  compression: SNAPPY
  encodings: ('PLAIN', 'RLE', 'BIT_PACKED')
  has_dictionary_page: False
  dictionary_page_offset: None
  data_page_offset: 4
  total_compressed_size: 13974153
  total_uncompressed_size: 24036999

country
  file_offset: 13974206
  file_path: 
  physical_type: BYTE_ARRAY
  num_values: 1848737
  path_in_schema: country
  is_stats_set: True
  statistics:
      has_min_max: True
      min: BR
      max: US
      null_count: 0
      distinct_count: None
      num_values: 1848737
      physical_type: BYTE_ARRAY
      logical_type: String
      converted_type (legacy):

In [33]:
for i in range(rg.num_columns):
    column = rg.column(i)

    print(column.path_in_schema)

    stats = column.statistics

    if stats is None:
        print("No statistics")
    else:
        print("Min:", stats.min)
        print("Max:", stats.max)
        print("Null Count:", stats.null_count)

    print()

user_id
Min: 100963605
Max: 95388363
Null Count: 0

country
Min: BR
Max: US
Null Count: 0

